# Gold: Análise e Enriquecimento dos Dados do Bitcoin

Este notebook corresponde à camada Gold da arquitetura Medallion, sendo responsável pela geração de métricas analíticas e indicadores de mercado a partir dos dados da camada Silver.

Nesta etapa, são aplicadas transformações analíticas utilizando Spark SQL e window functions para enriquecer os dados com indicadores financeiros:

1. Variação diária percentual: variação do preço em relação ao dia anterior.

2. Média móvel de 7 dias: média do preço dos últimos 7 dias.

3. Média móvel de 30 dias: média do preço dos últimos 30 dias.

4. Média móvel de 90 dias: média do preço dos últimos 90 dias

Os dados analíticos são persistidos no formato Delta Lake no container gold do ADLS Gen2, com overwrite a cada execução, garantindo que todas as métricas sejam sempre recalculadas com o histórico completo atualizado.

In [0]:
%pip install azure-storage-blob

In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("env", "prod")
dbutils.widgets.text("execution_date", "")

env            = dbutils.widgets.get("env")
execution_date = dbutils.widgets.get("execution_date")

print(f"Ambiente: {env}")
print(f"Data execução: {execution_date}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

storage_account = dbutils.secrets.get(scope='kv-scope-bitcoin', key='adls-storage-account-name')
storage_key     = dbutils.secrets.get(scope='kv-scope-bitcoin', key='adls-storage-key-bitcoin')

SILVER_PATH = f'abfss://silver@{storage_account}.dfs.core.windows.net/bitcoin/'
GOLD_PATH   = f'abfss://gold@{storage_account}.dfs.core.windows.net/bitcoin/'

print('Configuração concluída.')

In [0]:
try:

    # Leitura da Silver 
    df_silver = spark.read.format('delta') \
        .option(f'fs.azure.account.key.{storage_account}.dfs.core.windows.net', storage_key) \
        .load(SILVER_PATH)

    print(f'Registros na Silver: {df_silver.count()}')

    # Window functions 
    window_order = Window.orderBy('data')
    window_7d    = Window.orderBy('data').rowsBetween(-6,  0)
    window_30d   = Window.orderBy('data').rowsBetween(-29, 0)
    window_90d   = Window.orderBy('data').rowsBetween(-89, 0)

    df_gold = df_silver \
        .withColumn('preco_anterior',
            F.lag('preco_usd', 1).over(window_order)
        ) \
        .withColumn('variacao_diaria_pct',
            F.when(
                F.col('preco_anterior').isNotNull(),
                ((F.col('preco_usd') - F.col('preco_anterior')) / F.col('preco_anterior')) * 100
            ).otherwise(F.lit(None))
        ) \
        .withColumn('media_movel_7d',
            F.avg('preco_usd').over(window_7d)
        ) \
        .withColumn('media_movel_30d',
            F.avg('preco_usd').over(window_30d)
        ) \
        .withColumn('media_movel_90d',
            F.avg('preco_usd').over(window_90d)
        ) \
        .drop('preco_anterior')

    # Seleciona colunas finais 
    df_gold = df_gold.select(
        'data',
        'ativo',
        'moeda',
        'preco_usd',
        'market_cap_usd',
        'volume_usd',
        'variacao_diaria_pct',
        'media_movel_7d',
        'media_movel_30d',
        'media_movel_90d',
        'data_carga_lake',
        'fonte'
    ).orderBy('data')

    total_gold = df_gold.count()
    print(f'Registros na Gold: {total_gold}')
    df_gold.limit(10).display()

    # Grava em Delta Lake com overwrite
    df_gold.write.format('delta') \
        .mode('overwrite') \
        .option('overwriteSchema', 'true') \
        .option(f'fs.azure.account.key.{storage_account}.dfs.core.windows.net', storage_key) \
        .save(GOLD_PATH)

    # Registra tabela no catálogo 
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS gold.tb_bitcoin
        USING DELTA
        LOCATION '{GOLD_PATH}'
    """)

    print('Gold concluída com sucesso.')
    dbutils.notebook.exit(f'SUCCESS: Gold concluída. Total: {total_gold}')

except Exception as e:
    error_msg = str(e)
    if 'SUCCESS' in error_msg:
        dbutils.notebook.exit(error_msg)
    print(f'ERRO no Gold: {error_msg}')
    dbutils.notebook.exit(f'ERROR: Gold falhou. {error_msg}')